# Retail Sales and Inventory Operations Analysis

This notebook analyzes a simulated outdoor retail dataset at the store-date-product level. The goal is to identify sales drivers, inventory risks, product performance patterns, and operations recommendations.

**Data transparency:** this dataset is simulated for portfolio demonstration. It is not real The North Face or VF Corporation data.

## 1. Load Libraries and Data

The analysis uses pandas for data work and matplotlib for clean portfolio charts.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'retail_sales_inventory_cleaned.csv'

df = pd.read_csv(DATA_PATH, parse_dates=['date'])
df.head()

## 2. Data Quality Checks

Before analyzing performance, check whether the dataset has missing values, duplicate rows, and the right date format.

In [ ]:
print('Rows:', len(df))
print('Columns:', len(df.columns))
print('Duplicate rows:', df.duplicated().sum())
print('Missing values:', df.isna().sum().sum())
print('Date range:', df['date'].min().date(), 'to', df['date'].max().date())

## 3. Create Business Metrics

These metrics translate raw sales and traffic data into business meaning:

- **Revenue** shows sales dollars.
- **Conversion rate** shows how effectively customer visits become transactions.
- **Sell-through rate** shows how quickly inventory moves.
- **Low-stock flag** highlights potential lost sales risk.

In [ ]:
df['revenue_check'] = df['units_sold'] * df['unit_price']
df['conversion_rate_check'] = df['transactions'] / df['customer_visits']
df['sell_through_rate_check'] = df['units_sold'] / df['starting_inventory']

df[['revenue', 'revenue_check', 'conversion_rate', 'sell_through_rate', 'low_stock_flag']].head()

## 4. Executive KPI Snapshot

These top-line metrics summarize the simulated retail business.

In [ ]:
total_revenue = df['revenue'].sum()
total_units_sold = df['units_sold'].sum()
total_transactions = df['transactions'].sum()
total_customer_visits = df['customer_visits'].sum()
average_order_value = total_revenue / total_transactions
overall_conversion_rate = total_transactions / total_customer_visits

kpi_snapshot = pd.DataFrame({
    'Metric': ['Total Revenue', 'Total Units Sold', 'Average Order Value', 'Conversion Rate'],
    'Value': [
        f'${total_revenue:,.0f}',
        f'{total_units_sold:,.0f}',
        f'${average_order_value:,.0f}',
        f'{overall_conversion_rate:.1%}'
    ]
})
kpi_snapshot

## 5. Revenue by Month

Monthly revenue helps identify seasonality and planning periods for inventory, promotions, and staffing.

In [ ]:
monthly_revenue = (
    df.groupby('month', as_index=False)
    .agg(revenue=('revenue', 'sum'), units_sold=('units_sold', 'sum'))
    .sort_values('month')
)

plt.figure(figsize=(10, 5))
plt.plot(monthly_revenue['month'], monthly_revenue['revenue'], marker='o', linewidth=2.5)
plt.title('Monthly Revenue Trend')
plt.xlabel('Month')
plt.ylabel('Revenue')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.25)
plt.show()

monthly_revenue.sort_values('revenue', ascending=False).head()

## 6. Product Category Performance

Category-level revenue shows where the business earns most sales dollars. This supports merchandising and finance decisions.

In [ ]:
category_revenue = (
    df.groupby('product_category', as_index=False)
    .agg(revenue=('revenue', 'sum'), units_sold=('units_sold', 'sum'), average_sell_through_rate=('sell_through_rate', 'mean'))
    .sort_values('revenue', ascending=False)
)
category_revenue

In [ ]:
plt.figure(figsize=(9, 5))
plt.barh(category_revenue.sort_values('revenue')['product_category'], category_revenue.sort_values('revenue')['revenue'])
plt.title('Revenue by Product Category')
plt.xlabel('Revenue')
plt.ylabel('Product Category')
plt.grid(axis='x', alpha=0.25)
plt.show()

## 7. Product Ranking

Product ranking identifies the items that drive revenue and the items that drive unit volume.

In [ ]:
top_products_by_revenue = (
    df.groupby(['product_category', 'product_name'], as_index=False)
    .agg(revenue=('revenue', 'sum'), units_sold=('units_sold', 'sum'), average_sell_through_rate=('sell_through_rate', 'mean'))
    .sort_values('revenue', ascending=False)
)
top_products_by_revenue.head(10)

## 8. Inventory Risk and Low-Stock Products

Low-stock products can create lost sales risk. This table shows which products and stores need replenishment attention.

In [ ]:
low_stock_watchlist = (
    df[df['low_stock_flag']]
    .groupby(['store_name', 'product_category', 'product_name'], as_index=False)
    .agg(low_stock_days=('low_stock_flag', 'sum'), average_inventory_level=('inventory_level', 'mean'), units_sold=('units_sold', 'sum'), revenue=('revenue', 'sum'))
    .sort_values(['low_stock_days', 'revenue'], ascending=[False, False])
)
low_stock_watchlist.head(10)

## 9. Store Performance

Store-level performance connects revenue, traffic, conversion, and inventory risk. This supports operations and staffing decisions.

In [ ]:
store_performance = (
    df.groupby(['store_id', 'store_name', 'region'], as_index=False)
    .agg(revenue=('revenue', 'sum'), units_sold=('units_sold', 'sum'), transactions=('transactions', 'sum'), customer_visits=('customer_visits', 'sum'), low_stock_records=('low_stock_flag', 'sum'))
    .sort_values('revenue', ascending=False)
)
store_performance['conversion_rate'] = store_performance['transactions'] / store_performance['customer_visits']
store_performance

## 10. Business Recommendations

1. **Increase stock for fast-moving low-stock items.** Accessories and base layers with repeated low-stock days should be reviewed weekly during peak seasons.
2. **Protect premium outerwear inventory before winter.** Jackets generate the most revenue, so stock planning should happen before November and December.
3. **Promote slow-moving premium products selectively.** Use targeted campaigns and bundles instead of broad markdowns.
4. **Improve conversion through staffing.** Schedule more floor coverage during high-traffic periods and review lower-conversion stores for service gaps.
5. **Reduce lost sales risk.** Use a store-product low-stock report to trigger replenishment before products reach stockout levels.

## 11. Portfolio Takeaway

This project demonstrates the ability to clean retail data, calculate business KPIs, identify product and inventory risks, and translate findings into practical recommendations for retail operations, merchandising, finance, and marketing.